# DX 704 Week 11 Project

In this project, you will develop and test prompts asking a language model to classify text from a home services query and match it to an appropriate category of home services.

The full project description and a template notebook are available on GitHub: [Project 11 Materials](https://github.com/bu-cds-dx704/dx704-project-11).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1 : Design a Short Prompt

The provided file "queries.txt" contains sample text from requests by homeowners by email or phone.
These queries need to be classified as requesting an electrical, plumbing, or roofing or roofing services.
The provided file has columns query_id, query, and target_category.
Write a prompt template of 200 characters or less with parameter `query` for the homeowner query.
Your prompt should be suitable to use with the Python code `prompt_template.format(query=query)`.
Test your prompt with the model `gemini-2.0-flash` and suitable parsing code.

In [9]:
import pandas as pd
import google.generativeai as genai
import time
import os

# Configure API (load from environment variable)
import os
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))
model = genai.GenerativeModel('gemini-2.0-flash')

# Prompt template (200 characters or less)
prompt_template = "Classify as 'electrical', 'plumbing', or 'roofing': {query}\nAnswer with only the category."


df = pd.read_csv('queries.txt', sep='\t')

# Test function with rate limiting
def classify(query):
    prompt = prompt_template.format(query=query)
    response = model.generate_content(prompt)
    time.sleep(5)  # Wait 5 seconds between requests (12 per minute)
    return response.text.strip().lower()

# Run classification
df['prediction'] = df['query'].apply(classify)
df['correct'] = df['prediction'] == df['target_category'].str.lower()

# Save prompt template
with open('short-prompt.txt', 'w') as f:
    f.write(prompt_template)

# Save results
df[['query_id', 'prediction']].rename(columns={'prediction': 'predicted_category'}).to_csv('short-output.tsv', sep='\t', index=False)

# Results
accuracy = df['correct'].mean() * 100
print(f"Accuracy: {accuracy:.1f}%\n")
print(df[['query_id', 'target_category', 'prediction', 'correct']])
print(f"\nSaved: short-prompt.txt and short-output.tsv")

Classifying queries (this will take ~4 minutes for 50 queries)...
Accuracy: 100.0%

    query_id target_category  prediction  correct
0          1         roofing     roofing     True
1          2        plumbing    plumbing     True
2          3      electrical  electrical     True
3          4         roofing     roofing     True
4          5        plumbing    plumbing     True
5          6      electrical  electrical     True
6          7         roofing     roofing     True
7          8        plumbing    plumbing     True
8          9      electrical  electrical     True
9         10         roofing     roofing     True
10        11        plumbing    plumbing     True
11        12      electrical  electrical     True
12        13         roofing     roofing     True
13        14        plumbing    plumbing     True
14        15      electrical  electrical     True
15        16         roofing     roofing     True
16        17        plumbing    plumbing     True
17        18    

Save your prompt template in a file "short-prompt.txt".
Save the results of your prompt testing in "short-output.tsv" with columns `query_id` and `predicted_category`.

Submit "short-prompt.txt" and "short-output.tsv" in Gradescope.

Hint: your prompt may be re-tested with the Gemini API, so do not rely solely on lucky language model responses.

## Part 2: Find Short Prompt Mistakes

Construct 5 queries of 100 characters or less that trick your short prompt so that the wrong category is chosen.


In [ ]:
# Load the short prompt template
with open('short-prompt.txt', 'r') as f:
    prompt_template = f.read()

# Test function
def classify(query):
    prompt = prompt_template.format(query=query)
    response = model.generate_content(prompt)
    time.sleep(5)  # Rate limiting
    return response.text.strip().lower()

# Adversarial queries designed to trick the classifier
# Strategy: More ambiguous cases, edge cases, and misleading contexts
adversarial_queries = [
    {
        'query': 'Hi. Melissa came by and clogged my roof. Can you take a look at my sink?',
        'target_category': 'roofing'
    },
    {
        'query': 'Hi. Melissa came by and clogged my roof. Can you take a look at my sink?',
        'target_category': 'roofing'
    },
    {
        'query': 'Hi. Melissa came by and clogged my roof. Can you take a look at my sink?',
        'target_category': 'roofing'
    },
    {
        'query': 'Hi. Melissa came by and clogged my roof. Can you take a look at my sink?',
        'target_category': 'roofing'
    },
    {
        'query': 'Hi. Melissa came by and clogged my roof. Can you take a look at my sink?',
        'target_category': 'roofing'
    },
]

# Test each query
print("Testing adversarial queries...\n")
results = []
tricked = []

for item in adversarial_queries:
    predicted = classify(item['query'])
    is_wrong = predicted != item['target_category']
    
    if is_wrong:
        tricked.append({
            'query': item['query'],
            'target_category': item['target_category'],
            'predicted_category': predicted
        })
    
    status = 'TRICKED' if is_wrong else 'correct'
    print(f"{status}: {item['query'][:60]}...")
    print(f"  Target: {item['target_category']}, Predicted: {predicted}")
    print(f"  Strategy: {item['why']}\n")
    
    if len(tricked) >= 5:
        break

# If we got 5 mistakes, save them
if len(tricked) >= 5:
    df_mistakes = pd.DataFrame(tricked[:5])
    df_mistakes.to_csv('mistakes.tsv', sep='\t', index=False)
    print(f"\n✓ SUCCESS! Saved 5 adversarial queries to mistakes.tsv")
else:
    print(f"\nOnly found {len(tricked)} mistakes so far. Need {5-len(tricked)} more.")
    print("You may need to run with more queries or manually craft trickier ones.")

Testing adversarial queries...

TRICKED: Hi. Melissa came by and clogged my roof. Can you take a look...
  Target: roofing, Predicted: plumbing
  Strategy: Water pump strongly suggests plumbing context


Only found 1 mistakes so far. Need 4 more.
You may need to run with more queries or manually craft trickier ones.


Save your 5 queries in a file "mistakes.tsv" with columns `query`, `target_category` and `predicted_category`.

Submit "mistakes.tsv" in Gradescope.

## Part 3: Design a Long Prompt

Repeat part 1 with a length limit of 5000 characters.

In [ ]:
# YOUR CHANGES HERE

...

Save your longer prompt template in a file "long-prompt.txt".
Save the results of your prompt testing in "long-output.tsv".
Both files should use the same columns as part 1.

In [ ]:
# YOUR CHANGES HERE

...

Submit "long-prompt.txt" and "long-output.tsv" in Gradescope.

## Part 4: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 5: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.